# 🎒 VIT Campus Lost & Found: full-stack recovery platform

A single-notebook, full-stack app that replaces messy WhatsApp groups for the VIT community.

| Requirement | Where it lives |
|---|---|
| Official venues + category tags, separate Lost / Found feeds | Cell 2 (config), `/api/items` |
| Student auth, **public listings never expose reg. no. / e-mail / phone** | `/api/register`, alias-only `item_public()` |
| Private claim verification (finder-set challenge, Approve / Reject) | `/api/items/<id>/claims`, `/api/claims/<id>/decision` |
| In-app handoff thread + campus checkpoint agreement | `/api/claims/<id>/thread`, `/messages`, `/checkpoint` |
| Resolve by either party, removed from active boards | `/api/items/<id>/resolve` |
| **TensorFlow**: lost ↔ found match suggestions | Cell 4 (`TextVectorization` TF-IDF + cosine similarity) |
| Validation, error messages, loading / empty / submitted UI states | API handlers + embedded front-end |

**Stack:** Flask + SQLite (backend) · vanilla JS single-page UI served by Flask · TensorFlow (Keras `TextVectorization`, `tf.linalg`, `tf.matmul`).

**How to use:** Run cells top to bottom. The last server cell starts the app at <http://127.0.0.1:5000>. See `README.md` for running outside Jupyter.

## 1. Install dependencies

In [ ]:
# Install dependencies (skip if already installed). TensorFlow powers the lost<->found matcher.
%pip install -q flask werkzeug numpy tensorflow

## 2. Campus configuration & validation rules
Location tagging is restricted to the official landmarks below; the API rejects anything else.

In [2]:
import os, re, sqlite3, secrets, time, threading, json, tempfile
from datetime import datetime, timezone
from flask import Flask, g, jsonify, request, session, Response
from werkzeug.security import generate_password_hash, check_password_hash
from werkzeug.serving import make_server

# ---------------- Official VIT campus data (location tagging is restricted to these) ----------------
VENUE_GROUPS = {
    "Academic Blocks":     ["SJT", "TT", "PRP", "SMV", "MB", "GDN", "CDMM"],
    "Men's Hostel Blocks": [f"MH-{c}" for c in "ABCDEFGHIJKLMNOPQRST"],      # MH-A .. MH-T
    "Ladies' Hostel Blocks": [f"LH-{c}" for c in "ABCDEFGHIJ"],              # LH-A .. LH-J
    "Food Courts":         ["Gazebo", "Food Mall", "DC"],
    "Other Landmarks":     ["Central Library", "Sports Complex"],
}
VENUES = [v for group in VENUE_GROUPS.values() for v in group]
CATEGORIES = ["ID Cards", "Room Keys", "Calculators", "Lab Equipment", "Earphones", "Wallets"]

# Safe, staffed, public meetup points offered for handoff (edit freely for your campus).
CHECKPOINTS = [
    "SJT Ground Floor Reception",
    "TT Ground Floor Reception",
    "PRP Ground Floor Reception",
    "Central Library Security Desk",
    "Gazebo Food Court Entrance",
    "Food Mall Entrance",
    "Sports Complex Reception",
    "Men's Hostel Main Gate Security Desk",
    "Ladies' Hostel Main Gate Security Desk",
]

# ---------------- Validation patterns ----------------
EMAIL_DOMAIN = "@vitstudent.ac.in"
REGNO_RE = re.compile(r"^\d{2}[A-Z]{3}\d{4,5}$")                  # e.g. 24BCE1234
PHONE_RE = re.compile(r"^[6-9]\d{9}$")                             # Indian mobile
# Contact-info detectors (used to stop leaks in public text and chat)
FIND_PHONE = re.compile(r"(?<!\d)(?:\+?91[\s-]?)?[6-9]\d{4}[\s-]?\d{5}(?!\d)")
FIND_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
FIND_REGNO = re.compile(r"\b\d{2}[A-Za-z]{3}\d{4,5}\b")

DEFAULT_LOST_QUESTION = "Describe where and how you found this item, plus one detail only the finder would know."

app = Flask(__name__)
app.config["DATABASE"] = os.path.abspath("vit_recovery.db")
app.config["JSON_SORT_KEYS"] = False

_key_file = os.path.abspath(".secret_key")
if not os.path.exists(_key_file):
    with open(_key_file, "w") as f:
        f.write(secrets.token_hex(32))
app.secret_key = os.environ.get("SECRET_KEY") or open(_key_file).read().strip()
app.config.update(SESSION_COOKIE_HTTPONLY=True, SESSION_COOKIE_SAMESITE="Lax")
print(f"{len(VENUES)} venues, {len(CATEGORIES)} categories loaded.")

42 venues, 6 categories loaded.


## 3. Database bootstrap
Creates the SQLite schema (idempotent) and exports it as `schema.sql` for non-Jupyter bootstrapping.

In [3]:
SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS users (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    name          TEXT NOT NULL,
    reg_no        TEXT NOT NULL UNIQUE,
    email         TEXT NOT NULL UNIQUE,
    phone         TEXT NOT NULL,
    password_hash TEXT NOT NULL,
    alias         TEXT NOT NULL UNIQUE,          -- the ONLY identity ever shown to other students
    created_at    TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS items (
    id                    INTEGER PRIMARY KEY AUTOINCREMENT,
    kind                  TEXT NOT NULL CHECK (kind IN ('lost','found')),
    title                 TEXT NOT NULL,
    description           TEXT NOT NULL,
    category              TEXT NOT NULL,
    venue                 TEXT NOT NULL,
    verification_question TEXT NOT NULL,
    owner_id              INTEGER NOT NULL REFERENCES users(id),   -- the poster (finder or loser)
    status                TEXT NOT NULL DEFAULT 'open' CHECK (status IN ('open','resolved')),
    created_at            TEXT NOT NULL,
    resolved_at           TEXT,
    resolved_by           INTEGER REFERENCES users(id)
);
CREATE INDEX IF NOT EXISTS idx_items_feed ON items(status, kind, created_at DESC);

CREATE TABLE IF NOT EXISTS claims (
    id                   INTEGER PRIMARY KEY AUTOINCREMENT,
    item_id              INTEGER NOT NULL REFERENCES items(id),
    claimant_id          INTEGER NOT NULL REFERENCES users(id),
    answer               TEXT NOT NULL,
    status               TEXT NOT NULL DEFAULT 'pending'
                         CHECK (status IN ('pending','approved','rejected','closed')),
    checkpoint           TEXT,
    checkpoint_by        INTEGER REFERENCES users(id),
    checkpoint_confirmed INTEGER NOT NULL DEFAULT 0,
    created_at           TEXT NOT NULL,
    decided_at           TEXT,
    UNIQUE (item_id, claimant_id)
);

CREATE TABLE IF NOT EXISTS messages (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    claim_id   INTEGER NOT NULL REFERENCES claims(id),
    sender_id  INTEGER REFERENCES users(id),     -- NULL = system message
    body       TEXT NOT NULL,
    created_at TEXT NOT NULL
);
CREATE INDEX IF NOT EXISTS idx_messages_claim ON messages(claim_id, id);
"""

def bootstrap_db(path=None):
    """Create all tables (idempotent). Safe to run repeatedly."""
    path = path or app.config["DATABASE"]
    con = sqlite3.connect(path)
    con.executescript(SCHEMA_SQL)
    con.commit(); con.close()
    return path

def db():
    if "db" not in g:
        g.db = sqlite3.connect(app.config["DATABASE"])
        g.db.row_factory = sqlite3.Row
        g.db.execute("PRAGMA foreign_keys = ON")
    return g.db

@app.teardown_appcontext
def close_db(_exc):
    con = g.pop("db", None)
    if con is not None:
        con.close()

def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

# Also export the schema as a standalone script so the DB can be bootstrapped without Jupyter.
with open("schema.sql", "w") as f:
    f.write(SCHEMA_SQL)
print("Database ready at", bootstrap_db(), "| schema exported to schema.sql")

Database ready at C:\Users\Ameesha\Downloads\vit_recovery.db | schema exported to schema.sql


## 4. TensorFlow matcher
For a *lost* post it ranks open *found* posts (and vice versa): a TF-IDF `TextVectorization` layer embeds titles/descriptions, cosine similarity is computed with `tf.linalg.l2_normalize` + `tf.matmul`, then blended with category and venue agreement. Falls back to NumPy automatically if TensorFlow can't be imported.

In [4]:
# ---------------- TensorFlow-powered lost <-> found matcher ----------------
# Builds a TF-IDF text-vectorization layer over the candidate pool, embeds the query item,
# and ranks candidates by cosine similarity (tf.linalg.l2_normalize + tf.matmul),
# blended with category and venue agreement. Falls back to NumPy if TensorFlow is missing.
import numpy as np
try:
    import tensorflow as tf
    HAS_TF = True
except Exception as _e:                      # pragma: no cover
    HAS_TF = False
    print("TensorFlow not available, using NumPy fallback:", _e)

def _item_text(it):
    return f"{it['title']} {it['title']} {it['category']} {it['description']}".lower()

def _tfidf_matrix(texts):
    """Return an (n, vocab) TF-IDF float32 matrix for texts."""
    if HAS_TF:
        layer = tf.keras.layers.TextVectorization(
            max_tokens=4000, output_mode="tf_idf", ngrams=2, standardize="lower_and_strip_punctuation")
        layer.adapt(tf.data.Dataset.from_tensor_slices(texts).batch(64))
        return layer(tf.constant(texts)).numpy().astype("float32")
    # NumPy fallback: unigram TF-IDF
    toks = [re.findall(r"[a-z0-9]+", t) for t in texts]
    vocab = {w: i for i, w in enumerate(sorted({w for t in toks for w in t}))}
    m = np.zeros((len(texts), len(vocab)), dtype="float32")
    for r, t in enumerate(toks):
        for w in t:
            m[r, vocab[w]] += 1
    idf = np.log((1 + len(texts)) / (1 + (m > 0).sum(0))) + 1
    return m * idf

def _cosine(query_vec, cand_mat):
    if HAS_TF:
        q = tf.linalg.l2_normalize(tf.constant(query_vec[None, :]), axis=1)
        c = tf.linalg.l2_normalize(tf.constant(cand_mat), axis=1)
        return tf.matmul(q, c, transpose_b=True).numpy()[0]
    q = query_vec / (np.linalg.norm(query_vec) + 1e-9)
    c = cand_mat / (np.linalg.norm(cand_mat, axis=1, keepdims=True) + 1e-9)
    return c @ q

def suggest_matches(query_item, candidates, top_k=5, min_score=0.15):
    """Rank candidate items (dict-like with title/description/category/venue) against query_item."""
    if not candidates:
        return []
    texts = [_item_text(query_item)] + [_item_text(c) for c in candidates]
    mat = _tfidf_matrix(texts)
    sims = _cosine(mat[0], mat[1:])
    scored = []
    for c, s in zip(candidates, sims):
        score = 0.7 * float(max(s, 0)) + 0.2 * (c["category"] == query_item["category"]) \
                + 0.1 * (c["venue"] == query_item["venue"])
        if score >= min_score:
            scored.append((round(score, 3), c))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

print("Matcher backend:", "TensorFlow " + tf.__version__ if HAS_TF else "NumPy fallback")

TensorFlow not available, using NumPy fallback: No module named 'tensorflow'
Matcher backend: NumPy fallback


## 5. REST API
Auth, feeds, claim verification, private handoff thread, resolve, and match suggestions. Every endpoint validates input and returns clear JSON errors (`{error, fields}`).

In [5]:
# ---------------- Helpers: errors, validation, auth ----------------
class ApiError(Exception):
    def __init__(self, message, status=400, fields=None):
        self.message, self.status, self.fields = message, status, fields or {}

@app.errorhandler(ApiError)
def handle_api_error(e):
    return jsonify(error=e.message, fields=e.fields), e.status

@app.errorhandler(404)
def _404(_e):
    return jsonify(error="Not found."), 404

@app.errorhandler(405)
def _405(_e):
    return jsonify(error="Method not allowed."), 405

@app.errorhandler(Exception)
def _500(e):
    from werkzeug.exceptions import HTTPException
    if isinstance(e, HTTPException):
        return jsonify(error=e.description), e.code
    app.logger.exception(e)
    return jsonify(error="Something went wrong on our side. Please try again."), 500

@app.after_request
def security_headers(resp):
    resp.headers["X-Content-Type-Options"] = "nosniff"
    resp.headers["X-Frame-Options"] = "DENY"
    resp.headers["Referrer-Policy"] = "same-origin"
    resp.headers["Cache-Control"] = "no-store"
    return resp

def body():
    data = request.get_json(silent=True)
    if not isinstance(data, dict):
        raise ApiError("Request body must be valid JSON.", 400)
    return data

def s(data, key):
    v = data.get(key, "")
    return v.strip() if isinstance(v, str) else ""

def check(fields):
    if fields:
        raise ApiError("Please fix the highlighted fields.", 422, fields)

def no_contact_info(text, errors, key, allow_regno=False):
    if FIND_PHONE.search(text) or FIND_EMAIL.search(text) or (not allow_regno and FIND_REGNO.search(text)):
        errors[key] = ("Please don't include phone numbers, e-mails or registration numbers. "
                       "Use the private claim and chat flow instead.")

def login_required(fn):
    from functools import wraps
    @wraps(fn)
    def wrapper(*a, **kw):
        uid = session.get("uid")
        if not uid or not db().execute("SELECT 1 FROM users WHERE id=?", (uid,)).fetchone():
            session.clear()
            raise ApiError("Please log in to continue.", 401)
        return fn(*a, **kw)
    return wrapper

def uid():
    return session["uid"]

def new_alias():
    while True:
        a = "Anon-" + secrets.token_hex(2).upper()
        if not db().execute("SELECT 1 FROM users WHERE alias=?", (a,)).fetchone():
            return a

_attempts = {}   # naive in-memory login throttle: email -> (count, window_start)
def throttle_login(email):
    c, t = _attempts.get(email, (0, time.time()))
    if time.time() - t > 300:
        c, t = 0, time.time()
    if c >= 5:
        raise ApiError("Too many failed attempts. Try again in a few minutes.", 429)
    return c, t

# ---------------- Auth ----------------
@app.post("/api/register")
def register():
    d = body(); e = {}
    name, reg, email, phone, pw = s(d, "name"), s(d, "reg_no").upper(), s(d, "email").lower(), s(d, "phone"), d.get("password", "")
    if not (2 <= len(name) <= 60): e["name"] = "Enter your full name (2-60 characters)."
    if not REGNO_RE.match(reg): e["reg_no"] = "Use your VIT registration number, e.g. 24BCE1234."
    if not email.endswith(EMAIL_DOMAIN) or len(email) <= len(EMAIL_DOMAIN) or not re.match(r"^[\w.+-]+$", email[:-len(EMAIL_DOMAIN)]):
        e["email"] = f"Use your VIT student e-mail (ending {EMAIL_DOMAIN})."
    if not PHONE_RE.match(phone): e["phone"] = "Enter a valid 10-digit Indian mobile number."
    if not (isinstance(pw, str) and len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"\d", pw)):
        e["password"] = "Password needs 8+ characters with letters and numbers."
    check(e)
    try:
        cur = db().execute(
            "INSERT INTO users(name,reg_no,email,phone,password_hash,alias,created_at) VALUES(?,?,?,?,?,?,?)",
            (name, reg, email, phone, generate_password_hash(pw), new_alias(), now()))
        db().commit()
    except sqlite3.IntegrityError:
        raise ApiError("An account with this registration number or e-mail already exists.", 409,
                       {"email": "Already registered. Try logging in."})
    session.clear(); session["uid"] = cur.lastrowid
    return jsonify(me=me_payload(cur.lastrowid)), 201

@app.post("/api/login")
def login():
    d = body(); email = s(d, "email").lower(); pw = d.get("password", "")
    c, t = throttle_login(email)
    u = db().execute("SELECT * FROM users WHERE email=?", (email,)).fetchone()
    if not u or not isinstance(pw, str) or not check_password_hash(u["password_hash"], pw):
        _attempts[email] = (c + 1, t)
        raise ApiError("Incorrect e-mail or password.", 401)
    _attempts.pop(email, None)
    session.clear(); session["uid"] = u["id"]
    return jsonify(me=me_payload(u["id"]))

@app.post("/api/logout")
def logout():
    session.clear()
    return jsonify(ok=True)

def me_payload(user_id):
    u = db().execute("SELECT alias, name FROM users WHERE id=?", (user_id,)).fetchone()
    return {"alias": u["alias"], "name": u["name"]}   # own name only; never reg_no/phone/email

@app.get("/api/me")
def me():
    if not session.get("uid"):
        return jsonify(me=None)
    return jsonify(me=me_payload(uid()))

@app.get("/api/meta")
def meta():
    return jsonify(venue_groups=VENUE_GROUPS, categories=CATEGORIES, checkpoints=CHECKPOINTS)

# ---------------- Items (public feeds are alias-only) ----------------
def item_public(r, viewer_id=None):
    return {
        "id": r["id"], "kind": r["kind"], "title": r["title"], "description": r["description"],
        "category": r["category"], "venue": r["venue"], "poster": r["alias"],   # alias only
        "verification_question": r["verification_question"],
        "created_at": r["created_at"], "status": r["status"],
        "is_mine": viewer_id is not None and r["owner_id"] == viewer_id,
        "my_claim_status": r["my_claim"] if "my_claim" in r.keys() else None,
    }

@app.get("/api/items")
def list_items():
    kind = request.args.get("kind", "")
    category = request.args.get("category", "")
    venue = request.args.get("venue", "")
    q = request.args.get("q", "").strip()[:60]
    try:
        page = max(1, int(request.args.get("page", 1)))
    except ValueError:
        raise ApiError("Invalid page number.", 400)
    if kind not in ("lost", "found"): raise ApiError("kind must be 'lost' or 'found'.", 400)
    if category and category not in CATEGORIES: raise ApiError("Unknown category.", 400)
    if venue and venue not in VENUES: raise ApiError("Unknown campus location.", 400)

    viewer = session.get("uid") or -1
    sql = ("SELECT i.*, u.alias, (SELECT status FROM claims c WHERE c.item_id=i.id AND c.claimant_id=?) AS my_claim "
           "FROM items i JOIN users u ON u.id=i.owner_id WHERE i.status='open' AND i.kind=?")
    args = [viewer, kind]
    if category: sql += " AND i.category=?"; args.append(category)
    if venue:    sql += " AND i.venue=?";    args.append(venue)
    if q:
        like = "%" + q.replace("%", "").replace("_", "") + "%"
        sql += " AND (i.title LIKE ? OR i.description LIKE ?)"; args += [like, like]
    sql += " ORDER BY i.created_at DESC, i.id DESC LIMIT 31 OFFSET ?"; args.append((page - 1) * 30)
    rows = db().execute(sql, args).fetchall()
    return jsonify(items=[item_public(r, session.get("uid")) for r in rows[:30]], has_more=len(rows) > 30, page=page)

def get_item_row(item_id):
    r = db().execute("SELECT i.*, u.alias, (SELECT status FROM claims c WHERE c.item_id=i.id AND c.claimant_id=?) AS my_claim "
                     "FROM items i JOIN users u ON u.id=i.owner_id WHERE i.id=?", (session.get("uid") or -1, item_id)).fetchone()
    if not r: raise ApiError("Item not found.", 404)
    return r

@app.get("/api/items/<int:item_id>")
def item_detail(item_id):
    return jsonify(item=item_public(get_item_row(item_id), session.get("uid")))

@app.post("/api/items")
@login_required
def create_item():
    d = body(); e = {}
    kind, title, desc = s(d, "kind"), s(d, "title"), s(d, "description")
    category, venue, question = s(d, "category"), s(d, "venue"), s(d, "verification_question")
    if kind not in ("lost", "found"): e["kind"] = "Choose Lost or Found."
    if not (3 <= len(title) <= 80): e["title"] = "Title must be 3-80 characters."
    if not (10 <= len(desc) <= 500): e["description"] = "Description must be 10-500 characters."
    if category not in CATEGORIES: e["category"] = "Pick one of the listed categories."
    if venue not in VENUES: e["venue"] = "Pick an official VIT campus location."
    if kind == "found":
        if not (10 <= len(question) <= 200): e["verification_question"] = "Write a challenge question (10-200 characters)."
    else:
        question = DEFAULT_LOST_QUESTION
    for key, text in (("title", title), ("description", desc), ("verification_question", question if kind == "found" else "")):
        if text: no_contact_info(text, e, key)
    check(e)
    recent = db().execute("SELECT COUNT(*) c FROM items WHERE owner_id=? AND created_at > ?",
                          (uid(), datetime.fromtimestamp(time.time() - 3600, timezone.utc).isoformat(timespec="seconds"))).fetchone()["c"]
    if recent >= 10: raise ApiError("You're posting too fast. Please wait a bit.", 429)
    cur = db().execute("INSERT INTO items(kind,title,description,category,venue,verification_question,owner_id,created_at) "
                       "VALUES(?,?,?,?,?,?,?,?)", (kind, title, desc, category, venue, question, uid(), now()))
    db().commit()
    return jsonify(item=item_public(get_item_row(cur.lastrowid), uid())), 201

# ---------------- Claim verification workflow ----------------
@app.post("/api/items/<int:item_id>/claims")
@login_required
def submit_claim(item_id):
    item = get_item_row(item_id)
    if item["status"] != "open": raise ApiError("This item has already been resolved.", 409)
    if item["owner_id"] == uid(): raise ApiError("You can't claim your own post.", 403)
    answer = s(body(), "answer"); e = {}
    if not (3 <= len(answer) <= 300): e["answer"] = "Answer must be 3-300 characters."
    if FIND_PHONE.search(answer) or FIND_EMAIL.search(answer):
        e["answer"] = "Please don't include phone numbers or e-mails. The poster only needs the verification detail."
    check(e)
    try:
        cur = db().execute("INSERT INTO claims(item_id,claimant_id,answer,created_at) VALUES(?,?,?,?)",
                           (item_id, uid(), answer, now()))
        db().commit()
    except sqlite3.IntegrityError:
        raise ApiError("You've already submitted a claim for this item.", 409)
    return jsonify(claim={"id": cur.lastrowid, "status": "pending"}), 201

def load_claim(claim_id):
    c = db().execute("SELECT c.*, i.owner_id, i.title, i.kind, i.status AS item_status, i.venue, i.category "
                     "FROM claims c JOIN items i ON i.id=c.item_id WHERE c.id=?", (claim_id,)).fetchone()
    if not c: raise ApiError("Claim not found.", 404)
    return c

@app.get("/api/dashboard")
@login_required
def dashboard():
    con = db()
    mine = []
    for it in con.execute("SELECT i.*, u.alias FROM items i JOIN users u ON u.id=i.owner_id "
                          "WHERE i.owner_id=? ORDER BY i.status='resolved', i.created_at DESC", (uid(),)):
        claims = con.execute("SELECT c.id, c.answer, c.status, c.created_at, u.alias AS claimant FROM claims c "
                             "JOIN users u ON u.id=c.claimant_id WHERE c.item_id=? ORDER BY c.id", (it["id"],)).fetchall()
        d = item_public(it, uid()); d["claims"] = [dict(c) for c in claims]; mine.append(d)
    my_claims = [dict(r) for r in con.execute(
        "SELECT c.id, c.status, c.answer, c.checkpoint, c.checkpoint_confirmed, c.created_at, i.id AS item_id, i.title, i.kind, "
        "i.venue, i.status AS item_status, u.alias AS poster FROM claims c JOIN items i ON i.id=c.item_id "
        "JOIN users u ON u.id=i.owner_id WHERE c.claimant_id=? ORDER BY c.id DESC", (uid(),))]
    return jsonify(my_items=mine, my_claims=my_claims)

@app.post("/api/claims/<int:claim_id>/decision")
@login_required
def decide(claim_id):
    c = load_claim(claim_id)
    if c["owner_id"] != uid(): raise ApiError("Only the person who posted this item can review its claims.", 403)
    decision = s(body(), "decision")
    if decision not in ("approve", "reject"): raise ApiError("decision must be 'approve' or 'reject'.", 400)
    if c["status"] != "pending": raise ApiError("This claim has already been reviewed.", 409)
    if c["item_status"] != "open": raise ApiError("This item is already resolved.", 409)
    con = db()
    if decision == "approve":
        if con.execute("SELECT 1 FROM claims WHERE item_id=? AND status='approved'", (c["item_id"],)).fetchone():
            raise ApiError("Another claim is already approved for this item.", 409)
        con.execute("UPDATE claims SET status='approved', decided_at=? WHERE id=?", (now(), claim_id))
        con.execute("INSERT INTO messages(claim_id,sender_id,body,created_at) VALUES(?,NULL,?,?)",
                    (claim_id, "Claim approved. Chat here and agree on a campus checkpoint to meet. "
                               "Don't share phone numbers; this thread is private.", now()))
    else:
        con.execute("UPDATE claims SET status='rejected', decided_at=? WHERE id=?", (now(), claim_id))
    con.commit()
    return jsonify(status="approved" if decision == "approve" else "rejected")

# ---------------- Private handoff thread ----------------
def thread_access(claim_id):
    c = load_claim(claim_id)
    if uid() not in (c["owner_id"], c["claimant_id"]): raise ApiError("You don't have access to this thread.", 403)
    if c["status"] != "approved": raise ApiError("This thread opens once the claim is approved.", 409)
    return c

def role_of(c, user_id):
    if user_id == c["owner_id"]:
        return "Finder" if c["kind"] == "found" else "Owner"
    return "Claimant" if c["kind"] == "found" else "Finder"

@app.get("/api/claims/<int:claim_id>/thread")
@login_required
def thread(claim_id):
    c = thread_access(claim_id)
    msgs = db().execute("SELECT id, sender_id, body, created_at FROM messages WHERE claim_id=? ORDER BY id", (claim_id,)).fetchall()
    return jsonify(
        item={"id": c["item_id"], "title": c["title"], "venue": c["venue"], "kind": c["kind"], "status": c["item_status"]},
        me_role=role_of(c, uid()),
        checkpoint={"place": c["checkpoint"], "confirmed": bool(c["checkpoint_confirmed"]),
                    "proposed_by_me": c["checkpoint_by"] == uid() if c["checkpoint"] else None},
        messages=[{"id": m["id"], "mine": m["sender_id"] == uid(), "system": m["sender_id"] is None,
                   "role": "System" if m["sender_id"] is None else role_of(c, m["sender_id"]),
                   "body": m["body"], "created_at": m["created_at"]} for m in msgs])

@app.post("/api/claims/<int:claim_id>/messages")
@login_required
def post_message(claim_id):
    c = thread_access(claim_id)
    if c["item_status"] != "open": raise ApiError("This item is resolved; the thread is now read-only.", 409)
    text = s(body(), "body"); e = {}
    if not (1 <= len(text) <= 500): e["body"] = "Message must be 1-500 characters."
    elif FIND_PHONE.search(text) or FIND_EMAIL.search(text):
        e["body"] = "For your safety, phone numbers and e-mails can't be shared here. Agree on a campus checkpoint instead."
    check(e)
    last = db().execute("SELECT COUNT(*) n FROM messages WHERE sender_id=? AND created_at > ?",
                        (uid(), datetime.fromtimestamp(time.time() - 60, timezone.utc).isoformat(timespec="seconds"))).fetchone()["n"]
    if last >= 20: raise ApiError("Slow down a little and try again in a minute.", 429)
    db().execute("INSERT INTO messages(claim_id,sender_id,body,created_at) VALUES(?,?,?,?)", (claim_id, uid(), text, now()))
    db().commit()
    return jsonify(ok=True), 201

@app.post("/api/claims/<int:claim_id>/checkpoint")
@login_required
def propose_checkpoint(claim_id):
    c = thread_access(claim_id)
    if c["item_status"] != "open": raise ApiError("This item is resolved.", 409)
    place = s(body(), "checkpoint")
    if place not in CHECKPOINTS: raise ApiError("Choose one of the approved campus checkpoints.", 422, {"checkpoint": "Invalid checkpoint."})
    db().execute("UPDATE claims SET checkpoint=?, checkpoint_by=?, checkpoint_confirmed=0 WHERE id=?", (place, uid(), claim_id))
    db().execute("INSERT INTO messages(claim_id,sender_id,body,created_at) VALUES(?,NULL,?,?)",
                 (claim_id, f"{role_of(c, uid())} proposed meeting at: {place}", now()))
    db().commit()
    return jsonify(ok=True)

@app.post("/api/claims/<int:claim_id>/checkpoint/confirm")
@login_required
def confirm_checkpoint(claim_id):
    c = thread_access(claim_id)
    if not c["checkpoint"]: raise ApiError("No checkpoint has been proposed yet.", 409)
    if c["checkpoint_by"] == uid(): raise ApiError("The other person needs to confirm your proposal.", 403)
    if c["checkpoint_confirmed"]: raise ApiError("Checkpoint already confirmed.", 409)
    db().execute("UPDATE claims SET checkpoint_confirmed=1 WHERE id=?", (claim_id,))
    db().execute("INSERT INTO messages(claim_id,sender_id,body,created_at) VALUES(?,NULL,?,?)",
                 (claim_id, f"Meetup confirmed at {c['checkpoint']}. Meet in a public, staffed area.", now()))
    db().commit()
    return jsonify(ok=True)

# ---------------- Lifecycle: resolve ----------------
@app.post("/api/items/<int:item_id>/resolve")
@login_required
def resolve_item(item_id):
    item = get_item_row(item_id)
    if item["status"] == "resolved": raise ApiError("Already resolved.", 409)
    approved = db().execute("SELECT claimant_id FROM claims WHERE item_id=? AND status='approved'", (item_id,)).fetchone()
    allowed = item["owner_id"] == uid() or (approved and approved["claimant_id"] == uid())
    if not allowed: raise ApiError("Only the poster or the approved claimant can mark this as resolved.", 403)
    db().execute("UPDATE items SET status='resolved', resolved_at=?, resolved_by=? WHERE id=?", (now(), uid(), item_id))
    db().execute("UPDATE claims SET status='closed' WHERE item_id=? AND status='pending'", (item_id,))
    db().commit()
    return jsonify(status="resolved")

# ---------------- TensorFlow suggestions (poster only) ----------------
@app.get("/api/items/<int:item_id>/matches")
@login_required
def item_matches(item_id):
    item = get_item_row(item_id)
    if item["owner_id"] != uid(): raise ApiError("Only the poster can see suggested matches.", 403)
    if item["status"] != "open": raise ApiError("This item is resolved.", 409)
    other = "found" if item["kind"] == "lost" else "lost"
    cands = db().execute("SELECT i.*, u.alias FROM items i JOIN users u ON u.id=i.owner_id "
                         "WHERE i.status='open' AND i.kind=? AND i.owner_id!=? ORDER BY i.created_at DESC LIMIT 300",
                         (other, uid())).fetchall()
    ranked = suggest_matches(dict(item), [dict(c) for c in cands])
    return jsonify(backend="tensorflow" if HAS_TF else "numpy",
                   matches=[{"score": sc, **item_public(c, uid())} for sc, c in ranked])
print("API routes registered:", len([r for r in app.url_map.iter_rules() if r.rule.startswith('/api')]))

API routes registered: 17


## 6. Front-end (served at `/`)
Single-page UI with skeleton loaders, empty states, inline field errors, disabled/"Submitting…" buttons, toasts, a private dashboard and a live-polling chat.

In [6]:
INDEX_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>VIT Lost &amp; Found</title>
<style>
:root{--bg:#f5f6fa;--card:#fff;--ink:#1c2233;--muted:#6b7490;--line:#e3e6ef;--brand:#3b5bdb;--brand2:#2f49b3;
--lost:#e8590c;--found:#2b8a3e;--bad:#c92a2a;--radius:14px}
*{box-sizing:border-box}body{margin:0;font:15px/1.5 system-ui,-apple-system,Segoe UI,Roboto,sans-serif;background:var(--bg);color:var(--ink)}
header{position:sticky;top:0;z-index:5;background:var(--card);border-bottom:1px solid var(--line);display:flex;gap:8px;align-items:center;padding:10px 18px;flex-wrap:wrap}
.brand{font-weight:800;font-size:18px;margin-right:auto}.brand span{color:var(--brand)}
button{font:inherit;cursor:pointer;border:1px solid var(--line);background:var(--card);color:var(--ink);padding:8px 14px;border-radius:10px}
button:hover{border-color:var(--brand)}button.primary{background:var(--brand);color:#fff;border-color:var(--brand)}button.primary:hover{background:var(--brand2)}
button.good{background:var(--found);color:#fff;border-color:var(--found)}button.bad{background:#fff;color:var(--bad);border-color:var(--bad)}
button:disabled{opacity:.6;cursor:not-allowed}nav button.active{background:var(--ink);color:#fff;border-color:var(--ink)}
main{max-width:1050px;margin:0 auto;padding:20px 16px 60px}
.filters{display:flex;gap:10px;flex-wrap:wrap;margin:14px 0}.filters>*{flex:1 1 170px}
input,select,textarea{font:inherit;width:100%;padding:9px 11px;border:1px solid var(--line);border-radius:10px;background:#fff;color:var(--ink)}
textarea{min-height:84px;resize:vertical}input:focus,select:focus,textarea:focus{outline:2px solid #b6c3ff;border-color:var(--brand)}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(300px,1fr));gap:14px}
.card{background:var(--card);border:1px solid var(--line);border-radius:var(--radius);padding:16px;display:flex;flex-direction:column;gap:8px}
.card h3{margin:0;font-size:17px}.muted{color:var(--muted);font-size:13px}.row{display:flex;gap:8px;flex-wrap:wrap;align-items:center}
.pill{font-size:12px;padding:2px 9px;border-radius:99px;border:1px solid var(--line);background:#f1f3f9}
.pill.lost{background:#fff0e6;color:var(--lost);border-color:#ffd8bf}.pill.found{background:#e6f7ea;color:var(--found);border-color:#c0ebc9}
.pill.pending{background:#fff9db;color:#946500}.pill.approved{background:#e6f7ea;color:var(--found)}.pill.rejected,.pill.closed{background:#ffe3e3;color:var(--bad)}
.skel{height:150px;border-radius:var(--radius);background:linear-gradient(90deg,#eceff6 25%,#f7f8fc 37%,#eceff6 63%);background-size:400% 100%;animation:sh 1.3s infinite}
@keyframes sh{0%{background-position:100% 0}100%{background-position:0 0}}
.empty{text-align:center;padding:50px 10px;color:var(--muted);background:var(--card);border:1px dashed var(--line);border-radius:var(--radius)}
.empty b{display:block;font-size:18px;color:var(--ink);margin-bottom:6px}
.backdrop{position:fixed;inset:0;background:rgba(20,25,45,.5);display:flex;align-items:flex-start;justify-content:center;padding:30px 12px;overflow:auto;z-index:20}
.modal{background:var(--card);border-radius:var(--radius);width:100%;max-width:560px;padding:20px;display:flex;flex-direction:column;gap:12px}
.modal h2{margin:0}.field label{display:block;font-weight:600;font-size:13px;margin-bottom:4px}.err{color:var(--bad);font-size:13px;margin-top:3px}
.fieldbad input,.fieldbad select,.fieldbad textarea{border-color:var(--bad)}
.banner{padding:10px 12px;border-radius:10px;background:#fff5f5;color:var(--bad);border:1px solid #ffc9c9}
.info{padding:10px 12px;border-radius:10px;background:#edf2ff;border:1px solid #c5d0ff;font-size:14px}
#toasts{position:fixed;bottom:16px;left:50%;transform:translateX(-50%);display:flex;flex-direction:column;gap:8px;z-index:50}
.toast{background:var(--ink);color:#fff;padding:10px 16px;border-radius:10px}.toast.bad{background:var(--bad)}.toast.ok{background:var(--found)}
.chat{display:flex;flex-direction:column;gap:6px;max-height:300px;overflow:auto;padding:10px;background:#f7f8fc;border-radius:10px;border:1px solid var(--line)}
.msg{max-width:80%;padding:7px 11px;border-radius:12px;background:#fff;border:1px solid var(--line)}.msg.mine{align-self:flex-end;background:#dbe4ff;border-color:#bac8ff}
.msg.sys{align-self:center;background:#fff9db;border-color:#ffec99;font-size:13px;text-align:center;max-width:95%}.msg small{display:block;color:var(--muted);font-size:11px}
.tabs{display:flex;gap:8px;margin:14px 0}.section-title{margin:26px 0 10px;font-size:18px}
.claimbox{border:1px solid var(--line);border-radius:10px;padding:10px;background:#fafbff;display:flex;flex-direction:column;gap:6px}
.spin{display:inline-block;width:12px;height:12px;border:2px solid currentColor;border-right-color:transparent;border-radius:50%;animation:r .7s linear infinite;vertical-align:-2px;margin-right:6px}@keyframes r{to{transform:rotate(360deg)}}
</style></head>
<body>
<header>
  <div class="brand">VIT <span>Lost &amp; Found</span></div>
  <nav class="row" id="nav"></nav>
</header>
<main id="view"></main>
<div id="modal"></div><div id="toasts"></div>
<script>
const $ = (s, r=document) => r.querySelector(s);
const esc = s => String(s ?? "").replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
const state = {me:null, meta:null, view:"board", kind:"lost", filters:{category:"",venue:"",q:""}, timer:null, pollTimer:null};

class ApiErr extends Error { constructor(m, status, fields){ super(m); this.status=status; this.fields=fields||{}; } }
async function api(path, method="GET", payload){
  let res;
  try { res = await fetch(path, {method, headers:{"Content-Type":"application/json"}, body: payload ? JSON.stringify(payload) : undefined}); }
  catch(e){ throw new ApiErr("Can't reach the server. Check your connection and try again.", 0); }
  let data = {}; try { data = await res.json(); } catch(e){}
  if(!res.ok) throw new ApiErr(data.error || "Something went wrong.", res.status, data.fields);
  return data;
}
function toast(msg, type=""){ const t=document.createElement("div"); t.className="toast "+type; t.textContent=msg; $("#toasts").appendChild(t); setTimeout(()=>t.remove(), 3500); }
function ago(iso){ const s=(Date.now()-new Date(iso))/1000; if(s<60)return "just now"; if(s<3600)return Math.floor(s/60)+" min ago"; if(s<86400)return Math.floor(s/3600)+" h ago"; return Math.floor(s/86400)+" d ago"; }
function closeModal(){ $("#modal").innerHTML=""; clearInterval(state.pollTimer); }
function openModal(html){ $("#modal").innerHTML = `<div class="backdrop" onmousedown="if(event.target===this)closeModal()"><div class="modal">${html}</div></div>`; }
function needLogin(){ if(state.me) return false; toast("Please log in first.","bad"); authModal(); return true; }
async function busy(btn, label, fn){ const old=btn.innerHTML; btn.disabled=true; btn.innerHTML=`<span class="spin"></span>${label}`; try{ return await fn(); } finally{ btn.disabled=false; btn.innerHTML=old; } }
function fieldErrors(root, fields){ root.querySelectorAll(".err").forEach(e=>e.remove()); root.querySelectorAll(".fieldbad").forEach(e=>e.classList.remove("fieldbad"));
  for(const [k,v] of Object.entries(fields||{})){ const el=root.querySelector(`[name="${k}"]`); if(!el) continue; const f=el.closest(".field"); f.classList.add("fieldbad"); const d=document.createElement("div"); d.className="err"; d.textContent=v; f.appendChild(d); } }
const field = (label,name,input) => `<div class="field"><label>${label}</label>${input}</div>`;
const venueOptions = sel => Object.entries(state.meta.venue_groups).map(([g,vs])=>`<optgroup label="${esc(g)}">${vs.map(v=>`<option ${v===sel?"selected":""}>${esc(v)}</option>`).join("")}</optgroup>`).join("");
const catOptions = sel => state.meta.categories.map(c=>`<option ${c===sel?"selected":""}>${esc(c)}</option>`).join("");

function renderNav(){
  $("#nav").innerHTML = `
    <button data-k="lost" class="${state.view==='board'&&state.kind==='lost'?'active':''}">Lost</button>
    <button data-k="found" class="${state.view==='board'&&state.kind==='found'?'active':''}">Found</button>
    ${state.me?`<button data-k="dash" class="${state.view==='dash'?'active':''}">My dashboard</button>`:""}
    <button class="primary" data-k="report">+ Report item</button>
    ${state.me?`<span class="muted">${esc(state.me.name)} · ${esc(state.me.alias)}</span><button data-k="out">Log out</button>`:`<button data-k="in">Log in</button>`}`;
  $("#nav").querySelectorAll("button").forEach(b=>b.onclick=()=>{
    const k=b.dataset.k;
    if(k==="lost"||k==="found"){ state.view="board"; state.kind=k; go(); }
    else if(k==="dash"){ state.view="dash"; go(); }
    else if(k==="report"){ reportModal(); } else if(k==="in"){ authModal(); }
    else if(k==="out"){ api("/api/logout","POST").then(()=>{ state.me=null; state.view="board"; go(); toast("Logged out."); }); }
  });
}
function go(){ renderNav(); state.view==="dash" ? renderDashboard() : renderBoard(); }

/* ---------------- Board ---------------- */
function renderBoard(){
  const f=state.filters;
  $("#view").innerHTML = `
    <h1 style="margin:6px 0">${state.kind==="lost"?"Lost items":"Found items"} on campus</h1>
    <div class="muted">${state.kind==="lost"?"Someone is looking for these. Spotted one? Open it and tell the owner privately.":"Someone found these. Is one yours? Answer the finder's verification question to claim it."}</div>
    <div class="filters">
      <select id="fcat"><option value="">All categories</option>${catOptions(f.category)}</select>
      <select id="fven"><option value="">All locations</option>${venueOptions(f.venue)}</select>
      <input id="fq" placeholder="Search title or description" value="${esc(f.q)}">
    </div>
    <div id="grid" class="grid"></div><div id="more" style="text-align:center;margin-top:14px"></div>`;
  $("#fcat").onchange=e=>{f.category=e.target.value;loadItems();}; $("#fven").onchange=e=>{f.venue=e.target.value;loadItems();};
  $("#fq").oninput=e=>{f.q=e.target.value; clearTimeout(state.timer); state.timer=setTimeout(loadItems,300);};
  loadItems();
}
async function loadItems(page=1){
  const grid=$("#grid"); if(!grid) return;
  if(page===1) grid.innerHTML=`<div class="skel"></div><div class="skel"></div><div class="skel"></div>`;
  const p=new URLSearchParams({kind:state.kind,page,...Object.fromEntries(Object.entries(state.filters).filter(([,v])=>v))});
  try{
    const d=await api("/api/items?"+p);
    if(page===1) grid.innerHTML="";
    if(!d.items.length && page===1){ grid.innerHTML=`<div class="empty" style="grid-column:1/-1"><b>Nothing here yet</b>No ${state.kind} items match these filters. Be the first to report one.</div>`; }
    d.items.forEach(it=>grid.insertAdjacentHTML("beforeend", card(it)));
    grid.querySelectorAll("[data-open]").forEach(b=>b.onclick=()=>itemModal(+b.dataset.open));
    $("#more").innerHTML = d.has_more ? `<button id="moreb">Load more</button>` : "";
    if(d.has_more) $("#moreb").onclick=()=>loadItems(page+1);
  }catch(e){ grid.innerHTML=`<div class="banner" style="grid-column:1/-1">${esc(e.message)} <button onclick="loadItems()">Retry</button></div>`; }
}
const card = it => `<div class="card">
  <div class="row"><span class="pill ${it.kind}">${it.kind.toUpperCase()}</span><span class="pill">${esc(it.category)}</span>${it.is_mine?'<span class="pill">Your post</span>':""}${it.my_claim_status?`<span class="pill ${it.my_claim_status}">Claim ${it.my_claim_status}</span>`:""}</div>
  <h3>${esc(it.title)}</h3><div>${esc(it.description.length>140?it.description.slice(0,140)+"…":it.description)}</div>
  <div class="muted">📍 ${esc(it.venue)} · ${ago(it.created_at)} · posted by ${esc(it.poster)}</div>
  <button data-open="${it.id}">${it.kind==="found"?"View / claim":"View / I found it"}</button></div>`;

/* ---------------- Item detail + claim ---------------- */
async function itemModal(id){
  openModal(`<div class="skel"></div>`);
  let it; try{ it=(await api("/api/items/"+id)).item; }catch(e){ openModal(`<div class="banner">${esc(e.message)}</div><button onclick="closeModal()">Close</button>`); return; }
  const action = it.is_mine
    ? `<div class="info">This is your post. Review claims in <b>My dashboard</b>.</div><div class="row"><button id="mt">✨ Suggest matches (TensorFlow)</button></div><div id="matches"></div>`
    : it.my_claim_status ? `<div class="info">You already submitted a claim: <span class="pill ${it.my_claim_status}">${it.my_claim_status}</span>. Track it in My dashboard.</div>`
    : `<div class="info"><b>Verification challenge</b><br>${esc(it.verification_question)}</div>
       <form id="cf" class="field">${field("Your answer","answer",`<textarea name="answer" maxlength="300" placeholder="Be specific; only the real owner/finder would know this"></textarea>`)}
       <div id="cerr"></div><button class="primary" id="cs" style="margin-top:8px">Submit verification request</button></form>`;
  openModal(`<div class="row"><span class="pill ${it.kind}">${it.kind.toUpperCase()}</span><span class="pill">${esc(it.category)}</span></div>
    <h2>${esc(it.title)}</h2><div>${esc(it.description)}</div><div class="muted">📍 ${esc(it.venue)} · ${ago(it.created_at)} · posted by ${esc(it.poster)} (identity hidden)</div>
    ${action}<button onclick="closeModal()">Close</button>`);
  const cf=$("#cf"); if(cf) cf.onsubmit=async ev=>{ ev.preventDefault(); if(needLogin()) return;
    const btn=$("#cs"); try{ await busy(btn,"Submitting…",()=>api(`/api/items/${id}/claims`,"POST",{answer:cf.answer.value}));
      cf.outerHTML=`<div class="info">✅ Request submitted. The poster will review it privately. You'll see the result in My dashboard.</div>`; toast("Claim submitted","ok");
    }catch(e){ fieldErrors(cf,e.fields); if(!Object.keys(e.fields).length) $("#cerr").innerHTML=`<div class="banner">${esc(e.message)}</div>`; if(e.status===401) authModal(); } };
  const mt=$("#mt"); if(mt) mt.onclick=async()=>{ const box=$("#matches"); try{ const d=await busy(mt,"Matching…",()=>api(`/api/items/${id}/matches`));
      box.innerHTML = d.matches.length ? `<div class="muted" style="margin:8px 0">Ranked by ${d.backend} similarity</div>`+d.matches.map(m=>`<div class="claimbox"><b>${esc(m.title)}</b> <span class="pill">score ${m.score}</span><div class="muted">${esc(m.category)} · ${esc(m.venue)}</div></div>`).join("") : `<div class="empty">No similar ${it.kind==="lost"?"found":"lost"} items yet.</div>`;
    }catch(e){ box.innerHTML=`<div class="banner">${esc(e.message)}</div>`; } };
}

/* ---------------- Report ---------------- */
function reportModal(){
  if(needLogin()) return;
  openModal(`<h2>Report an item</h2><form id="rf" style="display:flex;flex-direction:column;gap:10px">
    ${field("I…","kind",`<select name="kind" id="rk"><option value="lost">lost something</option><option value="found">found something</option></select>`)}
    ${field("Title","title",`<input name="title" maxlength="80" placeholder="e.g. Casio fx-991EX calculator">`)}
    ${field("Description","description",`<textarea name="description" maxlength="500" placeholder="Colour, brand, marks. Don't include phone numbers or reg. numbers."></textarea>`)}
    ${field("Category","category",`<select name="category">${catOptions()}</select>`)}
    ${field("Campus location","venue",`<select name="venue">${venueOptions()}</select>`)}
    <div id="vq">${field("Verification challenge (only the real owner can answer)","verification_question",`<input name="verification_question" maxlength="200" placeholder="e.g. What name and branch are on the ID tag?">`)}</div>
    <div id="rerr"></div><div class="row"><button class="primary" id="rs">Post</button><button type="button" onclick="closeModal()">Cancel</button></div></form>`);
  const rf=$("#rf"), sync=()=>{ $("#vq").style.display = rf.kind.value==="found"?"":"none"; }; rf.kind.value=state.kind; rf.kind.onchange=sync; sync();
  rf.onsubmit=async ev=>{ ev.preventDefault(); const payload=Object.fromEntries(new FormData(rf)); if(payload.kind==="lost") delete payload.verification_question;
    try{ const d=await busy($("#rs"),"Posting…",()=>api("/api/items","POST",payload));
      toast("Posted ✓","ok"); closeModal(); state.view="board"; state.kind=d.item.kind; go();
    }catch(e){ fieldErrors(rf,e.fields); $("#rerr").innerHTML=`<div class="banner">${esc(e.message)}</div>`; } };
}

/* ---------------- Auth ---------------- */
function authModal(mode="login"){
  const reg = mode==="register";
  openModal(`<h2>${reg?"Create your account":"Log in"}</h2><form id="af" style="display:flex;flex-direction:column;gap:10px">
    ${reg?field("Full name","name",`<input name="name" autocomplete="name">`)+field("Registration number","reg_no",`<input name="reg_no" placeholder="24BCE1234">`)+field("Mobile (kept private)","phone",`<input name="phone" inputmode="numeric" maxlength="10">`):""}
    ${field("VIT e-mail","email",`<input name="email" type="email" placeholder="name.2024@vitstudent.ac.in" autocomplete="email">`)}
    ${field("Password","password",`<input name="password" type="password" autocomplete="${reg?"new":"current"}-password">`)}
    ${reg?`<div class="muted">Your registration number, e-mail and phone are never shown to other students. You appear as an anonymous alias.</div>`:""}
    <div id="aerr"></div><button class="primary" id="as">${reg?"Sign up":"Log in"}</button>
    <button type="button" id="sw">${reg?"I already have an account":"Create an account"}</button></form>`);
  $("#sw").onclick=()=>authModal(reg?"login":"register");
  const af=$("#af"); af.onsubmit=async ev=>{ ev.preventDefault();
    try{ const d=await busy($("#as"),reg?"Creating…":"Logging in…",()=>api(reg?"/api/register":"/api/login","POST",Object.fromEntries(new FormData(af))));
      state.me=d.me; closeModal(); toast("Welcome, "+d.me.name,"ok"); go();
    }catch(e){ fieldErrors(af,e.fields); $("#aerr").innerHTML=`<div class="banner">${esc(e.message)}</div>`; } };
}

/* ---------------- Dashboard ---------------- */
async function renderDashboard(){
  $("#view").innerHTML=`<h1 style="margin:6px 0">My dashboard</h1><div class="skel"></div>`;
  try{
    const d=await api("/api/dashboard");
    const mine=d.my_items.map(it=>`<div class="card"><div class="row"><span class="pill ${it.kind}">${it.kind.toUpperCase()}</span><span class="pill">${esc(it.category)}</span>
      <span class="pill ${it.status==='resolved'?'approved':''}">${it.status}</span></div><h3>${esc(it.title)}</h3><div class="muted">📍 ${esc(it.venue)} · ${ago(it.created_at)}</div>
      <div class="muted">Challenge: ${esc(it.verification_question)}</div>
      ${it.claims.length?it.claims.map(c=>`<div class="claimbox"><div class="row"><b>${esc(c.claimant)}</b><span class="pill ${c.status}">${c.status}</span><span class="muted">${ago(c.created_at)}</span></div>
        <div>“${esc(c.answer)}”</div>
        <div class="row">${c.status==="pending"&&it.status==="open"?`<button class="good" data-d="approve" data-c="${c.id}">Approve</button><button class="bad" data-d="reject" data-c="${c.id}">Reject</button>`:""}
        ${c.status==="approved"?`<button class="primary" data-t="${c.id}">Open handoff chat</button>`:""}</div></div>`).join(""):`<div class="muted">No claims yet.</div>`}
      <div class="row">${it.status==="open"?`<button data-r="${it.id}">Mark resolved</button><button data-m="${it.id}">✨ Suggest matches</button>`:""}</div></div>`).join("");
    const claims=d.my_claims.map(c=>`<div class="card"><div class="row"><span class="pill ${c.kind}">${c.kind.toUpperCase()}</span><span class="pill ${c.status}">${c.status}</span>${c.item_status==="resolved"?'<span class="pill approved">resolved</span>':""}</div>
      <h3>${esc(c.title)}</h3><div class="muted">📍 ${esc(c.venue)} · posted by ${esc(c.poster)}</div><div>Your answer: “${esc(c.answer)}”</div>
      ${c.status==="approved"?`<button class="primary" data-t="${c.id}">Open handoff chat</button>${c.item_status==="open"?`<button data-r="${c.item_id}">Mark resolved</button>`:""}`:c.status==="pending"?`<div class="muted">Waiting for the poster to review.</div>`:""}</div>`).join("");
    $("#view").innerHTML=`<h1 style="margin:6px 0">My dashboard</h1>
      <h2 class="section-title">My posts &amp; incoming claims</h2><div class="grid">${mine||`<div class="empty" style="grid-column:1/-1"><b>No posts yet</b>Use “Report item” to post something lost or found.</div>`}</div>
      <h2 class="section-title">My claims</h2><div class="grid">${claims||`<div class="empty" style="grid-column:1/-1"><b>No claims yet</b>Open a Found item and answer its verification challenge.</div>`}</div>`;
    $("#view").querySelectorAll("[data-d]").forEach(b=>b.onclick=()=>busy(b,"Saving…",async()=>{ try{ await api(`/api/claims/${b.dataset.c}/decision`,"POST",{decision:b.dataset.d}); toast(b.dataset.d==="approve"?"Approved. Handoff chat is open.":"Rejected.","ok"); renderDashboard(); }catch(e){ toast(e.message,"bad"); } }));
    $("#view").querySelectorAll("[data-t]").forEach(b=>b.onclick=()=>threadModal(+b.dataset.t));
    $("#view").querySelectorAll("[data-r]").forEach(b=>b.onclick=()=>resolveItem(+b.dataset.r, b));
    $("#view").querySelectorAll("[data-m]").forEach(b=>b.onclick=()=>itemModal(+b.dataset.m));
  }catch(e){ $("#view").innerHTML=`<div class="banner">${esc(e.message)} <button onclick="renderDashboard()">Retry</button></div>`; }
}
async function resolveItem(id, btn){
  if(!confirm("Mark as resolved? It will be removed from the campus boards.")) return;
  try{ await busy(btn||document.body,"Saving…",()=>api(`/api/items/${id}/resolve`,"POST")); toast("Marked resolved ✓","ok"); closeModal(); go(); }catch(e){ toast(e.message,"bad"); }
}

/* ---------------- Handoff thread ---------------- */
async function threadModal(claimId){
  openModal(`<div class="skel"></div>`);
  const draw=async(keepInput)=>{
    let d; try{ d=await api(`/api/claims/${claimId}/thread`); }catch(e){ openModal(`<div class="banner">${esc(e.message)}</div><button onclick="closeModal()">Close</button>`); return; }
    const sig=JSON.stringify([d.messages.length,d.checkpoint,d.item.status]);
    if(keepInput && sig===draw.sig) return; draw.sig=sig;
    const draft=keepInput?($("#tm")?.value||""):""; const open=d.item.status==="open", cp=d.checkpoint;
    const cpBox = cp.place ? `<div class="info">📍 <b>${esc(cp.place)}</b> ${cp.confirmed?"✅ confirmed":cp.proposed_by_me?"(waiting for the other person to confirm)":"(proposed by the other person)"}
      ${open&&!cp.confirmed&&cp.proposed_by_me===false?`<button class="good" id="cc">Confirm</button>`:""}</div>` : `<div class="info">No meetup point agreed yet. Propose a staffed campus checkpoint.</div>`;
    const atBottom=()=>{ const c=$("#chat"); return !c || c.scrollHeight-c.scrollTop-c.clientHeight<40; }; const stick=atBottom();
    $("#modal").innerHTML=`<div class="backdrop"><div class="modal"><div class="row"><h2 style="margin-right:auto">Handoff: ${esc(d.item.title)}</h2><button onclick="closeModal()">✕</button></div>
      <div class="muted">You are the <b>${d.me_role}</b>. Identities stay hidden; share nothing personal.</div>${cpBox}
      ${open?`<div class="row"><select id="cps" style="flex:1">${state.meta.checkpoints.map(c=>`<option ${c===cp.place?"selected":""}>${esc(c)}</option>`).join("")}</select><button id="cpb">Propose</button></div>`:`<div class="info">Resolved. This thread is read-only.</div>`}
      <div class="chat" id="chat">${d.messages.map(m=>`<div class="msg ${m.system?"sys":m.mine?"mine":""}">${m.system?"":`<small>${m.mine?"You":esc(m.role)} · ${ago(m.created_at)}</small>`}${esc(m.body)}</div>`).join("")}</div>
      ${open?`<form id="mf" class="field"><div class="row"><input id="tm" name="body" maxlength="500" placeholder="Type a message" style="flex:1" value="${esc(draft)}"><button class="primary" id="ms">Send</button></div><div id="merr"></div></form>
      <button class="good" id="rv">✔ Item returned. Mark resolved</button>`:""}</div></div>`;
    if(stick){ const c=$("#chat"); c.scrollTop=c.scrollHeight; }
    if(!open) return;
    $("#cpb").onclick=async e=>{ try{ await busy(e.target,"Saving…",()=>api(`/api/claims/${claimId}/checkpoint`,"POST",{checkpoint:$("#cps").value})); draw(); }catch(x){ toast(x.message,"bad"); } };
    const cc=$("#cc"); if(cc) cc.onclick=async e=>{ try{ await busy(e.target,"Confirming…",()=>api(`/api/claims/${claimId}/checkpoint/confirm`,"POST")); draw(); }catch(x){ toast(x.message,"bad"); } };
    $("#mf").onsubmit=async ev=>{ ev.preventDefault(); try{ await busy($("#ms"),"Sending…",()=>api(`/api/claims/${claimId}/messages`,"POST",{body:$("#tm").value})); $("#tm").value=""; draw(); }
      catch(x){ $("#merr").innerHTML=`<div class="err">${esc(x.fields.body||x.message)}</div>`; } };
    $("#rv").onclick=()=>resolveItem(d.item.id,$("#rv"));
  };
  await draw();
  clearInterval(state.pollTimer); state.pollTimer=setInterval(()=>{ if($("#chat")) draw(true); else clearInterval(state.pollTimer); }, 5000);
}

(async function init(){
  try{ state.meta=await api("/api/meta"); state.me=(await api("/api/me")).me; }
  catch(e){ $("#view").innerHTML=`<div class="banner">${esc(e.message)}</div>`; return; }
  go();
})();
</script></body></html>"""

@app.get("/")
def index():
    return Response(INDEX_HTML, mimetype="text/html")
print("Frontend ready:", len(INDEX_HTML) // 1024, "KB")

Frontend ready: 24 KB


## 7. Self-test
Runs the full workflow on a temporary database and asserts the privacy guarantees: no personal data in public feeds, claimants can't see the finder, thread locked until approval, contact-info blocked in chat, resolve closes the thread.

In [7]:
# ---------------- End-to-end self-test (runs against a throw-away database) ----------------
def run_self_test():
    real_db = app.config["DATABASE"]
    tmp = os.path.join(tempfile.mkdtemp(), "test.db")
    app.config["DATABASE"] = tmp; bootstrap_db(tmp)
    try:
        finder, owner, thief = app.test_client(), app.test_client(), app.test_client()
        def reg(c, n, r, e, p):
            return c.post("/api/register", json=dict(name=n, reg_no=r, email=e, phone=p, password="Passw0rd123"))
        assert reg(finder, "Fiona Finder", "24BCE1001", "fiona@vitstudent.ac.in", "9876543210").status_code == 201
        assert reg(owner,  "Omar Owner",   "24BCE1002", "omar@vitstudent.ac.in",  "9876543211").status_code == 201
        assert reg(thief,  "Tara Thief",   "24BCE1003", "tara@vitstudent.ac.in",  "9876543212").status_code == 201

        # validation errors
        r = reg(app.test_client(), "X", "bad", "x@gmail.com", "123"); assert r.status_code == 422 and len(r.json["fields"]) >= 4
        assert reg(finder, "Fiona Finder", "24BCE1001", "fiona@vitstudent.ac.in", "9876543210").status_code == 409
        assert finder.post("/api/items", json=dict(kind="found", title="ID", description="x", category="Pets", venue="Mars")).status_code == 422
        assert finder.post("/api/items", json=dict(kind="found", title="ID card", description="Blue lanyard, call 9876543210 now",
                           category="ID Cards", venue="SJT", verification_question="What name is on the ID?")).status_code == 422
        assert app.test_client().post("/api/items", json={}).status_code == 401

        # finder posts a found ID card
        r = finder.post("/api/items", json=dict(kind="found", title="VIT ID card near SJT lift", description="Found an ID card on the ground floor, blue lanyard attached.",
                        category="ID Cards", venue="SJT", verification_question="What name and branch are printed on the ID tag?"))
        assert r.status_code == 201, r.json; item_id = r.json["item"]["id"]

        # privacy: public feed never leaks personal data
        feed = owner.get("/api/items?kind=found")
        assert feed.status_code == 200 and len(feed.json["items"]) == 1
        raw = feed.get_data(as_text=True)
        for secret in ("24BCE1001", "fiona@vitstudent.ac.in", "9876543210", "Fiona"):
            assert secret not in raw, f"LEAK: {secret}"
        assert feed.json["items"][0]["poster"].startswith("Anon-")
        assert owner.get("/api/items?kind=found&venue=Mars").status_code == 400

        # claims: thief answers wrong, owner answers right
        assert finder.post(f"/api/items/{item_id}/claims", json=dict(answer="whatever")).status_code == 403   # own post
        c1 = thief.post(f"/api/items/{item_id}/claims", json=dict(answer="Tara, Mech")); assert c1.status_code == 201
        assert thief.post(f"/api/items/{item_id}/claims", json=dict(answer="again")).status_code == 409
        c2 = owner.post(f"/api/items/{item_id}/claims", json=dict(answer="Omar Owner, CSE (BCE)")); assert c2.status_code == 201
        cid_thief, cid_owner = c1.json["claim"]["id"], c2.json["claim"]["id"]

        # claimant cannot review claims; thread locked until approval
        assert owner.post(f"/api/claims/{cid_owner}/decision", json=dict(decision="approve")).status_code == 403
        assert owner.get(f"/api/claims/{cid_owner}/thread").status_code == 409

        # finder reviews privately
        dash = finder.get("/api/dashboard").json
        answers = {c["claimant"]: c["answer"] for c in dash["my_items"][0]["claims"]}
        assert len(answers) == 2 and all(k.startswith("Anon-") for k in answers)
        assert "24BCE1002" not in json.dumps(dash)
        assert finder.post(f"/api/claims/{cid_thief}/decision", json=dict(decision="reject")).status_code == 200
        assert finder.post(f"/api/claims/{cid_owner}/decision", json=dict(decision="approve")).status_code == 200
        assert thief.get(f"/api/claims/{cid_thief}/thread").status_code in (403, 409)

        # handoff thread
        t = owner.get(f"/api/claims/{cid_owner}/thread"); assert t.status_code == 200 and t.json["me_role"] == "Claimant"
        assert owner.post(f"/api/claims/{cid_owner}/messages", json=dict(body="Text me on 9876543211")).status_code == 422
        assert owner.post(f"/api/claims/{cid_owner}/messages", json=dict(body="Thanks! Can we meet tomorrow?")).status_code == 201
        assert owner.post(f"/api/claims/{cid_owner}/checkpoint", json=dict(checkpoint="Mars Base")).status_code == 422
        assert owner.post(f"/api/claims/{cid_owner}/checkpoint", json=dict(checkpoint="SJT Ground Floor Reception")).status_code == 200
        assert owner.post(f"/api/claims/{cid_owner}/checkpoint/confirm").status_code == 403   # can't confirm own proposal
        assert finder.post(f"/api/claims/{cid_owner}/checkpoint/confirm").status_code == 200
        t = finder.get(f"/api/claims/{cid_owner}/thread").json
        assert t["checkpoint"]["confirmed"] and t["me_role"] == "Finder"

        # lifecycle: resolve removes from boards, thread becomes read-only
        assert thief.post(f"/api/items/{item_id}/resolve").status_code == 403
        assert owner.post(f"/api/items/{item_id}/resolve").status_code == 200       # approved claimant may resolve
        assert owner.get("/api/items?kind=found").json["items"] == []
        assert owner.post(f"/api/claims/{cid_owner}/messages", json=dict(body="hello")).status_code == 409
        assert finder.post(f"/api/items/{item_id}/resolve").status_code == 409

        # lost item flow + TensorFlow/NumPy matcher
        lost = owner.post("/api/items", json=dict(kind="lost", title="Casio fx-991EX calculator", description="Black scientific calculator with a green sticker, lost after lab.",
                          category="Calculators", venue="MB")); assert lost.status_code == 201
        finder.post("/api/items", json=dict(kind="found", title="Black Casio calculator", description="Scientific calculator with green sticker found in the lab.",
                    category="Calculators", venue="MB", verification_question="Which sticker colour is on the back?"))
        finder.post("/api/items", json=dict(kind="found", title="Hostel room key", description="Single brass key with a red tag near the block entrance.",
                    category="Room Keys", venue="MH-C", verification_question="What number is on the tag?"))
        m = owner.get(f"/api/items/{lost.json['item']['id']}/matches"); assert m.status_code == 200
        assert m.json["matches"] and "calculator" in m.json["matches"][0]["title"].lower()
        print("Top match:", m.json["matches"][0]["title"], "| score", m.json["matches"][0]["score"], "| backend:", m.json["backend"])
        print("ALL TESTS PASSED ✅")
    finally:
        app.config["DATABASE"] = real_db

run_self_test()

Top match: Black Casio calculator | score 0.75 | backend: numpy
ALL TESTS PASSED ✅


## 8. Launch the app
Starts the server in a background thread. Open **http://127.0.0.1:5000**. Call `stop_server()` to stop it.

In [8]:
# ---------------- Run the server inside the notebook ----------------
_server = {"srv": None, "thread": None}

def start_server(host="127.0.0.1", port=5000):
    if _server["srv"]:
        print("Already running."); return
    bootstrap_db()
    srv = make_server(host, port, app, threaded=True)
    t = threading.Thread(target=srv.serve_forever, daemon=True)
    t.start(); _server.update(srv=srv, thread=t)
    print(f"VIT Lost & Found running at http://{host}:{port}  (call stop_server() to stop)")

def stop_server():
    if _server["srv"]:
        _server["srv"].shutdown(); _server.update(srv=None, thread=None); print("Server stopped.")

start_server()
# Open http://127.0.0.1:5000 in your browser.

VIT Lost & Found running at http://127.0.0.1:5000  (call stop_server() to stop)


127.0.0.1 - - [08/Oct/2026 12:08:29] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [08/Oct/2026 12:08:29] "GET /api/meta HTTP/1.1" 200 -
127.0.0.1 - - [08/Oct/2026 12:08:29] "GET /api/me HTTP/1.1" 200 -
127.0.0.1 - - [08/Oct/2026 12:08:29] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [08/Oct/2026 12:08:29] "GET /api/items?kind=lost&page=1 HTTP/1.1" 200 -
127.0.0.1 - - [08/Oct/2026 12:08:33] "GET /api/items?kind=lost&page=1&category=ID+Cards HTTP/1.1" 200 -
127.0.0.1 - - [08/Oct/2026 12:08:35] "GET /api/items?kind=lost&page=1&category=ID+Cards&venue=PRP HTTP/1.1" 200 -
127.0.0.1 - - [08/Oct/2026 12:08:38] "GET /api/items?kind=lost&page=1&category=ID+Cards&venue=Gazebo HTTP/1.1" 200 -
127.0.0.1 - - [08/Oct/2026 12:08:48] "GET /api/items?kind=lost&page=1&category=ID+Cards HTTP/1.1" 200 -
127.0.0.1 - - [08/Oct/2026 12:10:31] "POST /api/register HTTP/1.1" 201 -
127.0.0.1 - - [08/Oct/2026 12:10:31] "GET /api/items?kind=lost&page=1&category=ID+Cards HTTP/1.1" 200 -
127.0.0.1 - - [08/Oct/2026 12:11: